In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


df = pd.read_csv(r"C:\Users\sonia\Downloads\internship\interenship.2\project2\train.csv")
df.head(5)
print(df.shape)
print(df.info())
print(df.describe())

missing = df.isnull().sum()


print(missing.sort_values())

df.drop(['PoolQC','MiscFeature','Alley','Fence'],
    axis=1,
    inplace=True)



# MasVnrType
df['MasVnrType'] = df['MasVnrType'].fillna('None')

# FireplaceQu
df['FireplaceQu'] = df['FireplaceQu'].fillna('No Fireplace')

# LotFrontage
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

# Garage columns
garage_cols = ['GarageType',
               'GarageFinish',
               'GarageQual',
               'GarageCond']

for col in garage_cols:
    df[col] = df[col].fillna('No Garage')

df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

# Basement columns
bsmt_cols = ['BsmtExposure',
             'BsmtFinType2',
             'BsmtQual',
             'BsmtCond',
             'BsmtFinType1']

for col in bsmt_cols:
    df[col] = df[col].fillna('No Basement')

df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

df['Electrical'] = df['Electrical'].fillna(
    df['Electrical'].mode()[0]
    
)
df.isnull().sum().sum()

corr = df.corr(numeric_only=True)

sale_corr = corr["SalePrice"].sort_values(ascending=False)

print(sale_corr.head(10))

plt.figure(figsize=(10,6))

sale_corr.head(10).plot(kind='bar')

plt.title("Top Features Correlated with SalePrice")

plt.show()


df = pd.get_dummies(df, drop_first=True)

X = df.drop("SalePrice", axis=1)

y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# model1
lr = LinearRegression()

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, pred_lr)

rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))

r2_lr = r2_score(y_test, pred_lr)

print(mae_lr, rmse_lr, r2_lr)

# model 2
dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)

mae_dt = mean_absolute_error(y_test, pred_dt)

rmse_dt = np.sqrt(mean_squared_error(y_test, pred_dt))

r2_dt = r2_score(y_test, pred_dt)

# model3
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, pred_rf)

rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))

r2_rf = r2_score(y_test, pred_rf)

comparison = pd.DataFrame({

"Model":[
"Linear Regression",
"Decision Tree",
"Random Forest"
],

"MAE":[
mae_lr,
mae_dt,
mae_rf
],

"RMSE":[
rmse_lr,
rmse_dt,
rmse_rf
],

"R2 Score":[
r2_lr,
r2_dt,
r2_rf
]

})

comparison

comparison.sort_values(
    by="R2 Score",
    ascending=False
)

residuals = y_test - pred_rf

plt.figure(figsize=(8,6))

plt.scatter(pred_rf, residuals)

plt.axhline(y=0)

plt.xlabel("Predicted Price")

plt.ylabel("Residuals")

plt.title("Residual Plot - Random Forest")

plt.show()

import pickle

with open('random_forest_model.pkl', 'wb') as file:
    pickle.dump(rf, file)
    
with open('linear_regression_model.pkl', 'wb') as file:
    pickle.dump(lr, file)
 
with open('decision_tree_model.pkl', 'wb') as file:
    pickle.dump(dt, file)  
  
with open('random_forest_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

prediction = loaded_model.predict(X_test)

print(prediction[:5])  
%pip install google-colab

try:
    from google.colab import files
    files.download('random_forest_model.pkl')
    files.download('linear_regression_model.pkl')
    files.download('decision_tree_model.pkl')
except Exception:
    print("Google Colab download skipped; model files were already saved locally.")

for file in os.listdir():
    print(file)
    
random_forest_residual_plot = "random_forest_residual_plot.png"

plt.figure(figsize=(8, 6))
plt.scatter(pred_rf, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.title("Residual Plot - Random Forest")
plt.tight_layout()
plt.savefig(random_forest_residual_plot)
plt.show()
df.to_csv("house_clean.csv", index=False)

print("Project Completed Successfully")
print("Best Model: Random Forest Regressor")
print("R2 Score:", r2_rf)
